# 04. IPW & Doubly Robust Estimation

## Background

Propensity score matching targets ATT on the matched sample — a subset of units. Inverse Probability Weighting (IPW) takes a different route: reweight each observation so the weighted sample mimics a randomized experiment. Treated units with low propensity scores represent many similar units who weren't treated; upweighting them fills in the counterfactual population.

The key insight: IPW is unbiased if the propensity score model is correctly specified. Outcome regression (direct adjustment) is unbiased if the outcome model is correctly specified. **Augmented IPW (AIPW)** — also called Doubly Robust estimation — is unbiased if *either* model is correct. This doubly robust property makes AIPW the estimator of choice for most observational studies in modern practice.

## What You'll Learn

- Horvitz-Thompson IPW estimator: E[Y(1)/e(X)] − E[Y(0)/(1−e(X))]
- Stabilized weights: SIPW — avoids extreme weights blowing up variance
- Outcome regression: direct covariate adjustment
- AIPW: combining propensity and outcome models for double robustness
- Variance estimation via bootstrap
- Diagnosing extreme weights

## Why This Matters

AIPW (Robins, Rotnitzky & Zhao 1994; Lunceford & Davidian 2004) is the foundation of modern causal ML. Double ML (notebook 08) and targeted maximum likelihood estimation (TMLE) are both refinements of the doubly robust principle. If you understand AIPW, you understand why cross-fitting matters and what "nuisance model" means in this context.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
def simulate_study(n=3000, seed=0):
    rng = np.random.default_rng(seed)
    age      = rng.normal(50, 12, n)
    severity = rng.normal(5, 2, n)
    bmi      = rng.normal(27, 5, n)
    smoke    = rng.binomial(1, 0.3, n)
    female   = rng.binomial(1, 0.5, n)

    logit = -3 + 0.06*age + 0.4*severity - 0.02*bmi + 0.3*smoke
    ps_true = 1 / (1 + np.exp(-logit))
    treated = rng.binomial(1, ps_true).astype(bool)

    y0 = 20 - 0.8*severity + 0.1*age - 0.2*bmi + rng.normal(0, 2, n)
    y1 = y0 + 3.0
    y_obs = np.where(treated, y1, y0)

    return pd.DataFrame({
        'treated': treated, 'age': age, 'severity': severity,
        'bmi': bmi, 'smoke': smoke, 'female': female,
        'y_obs': y_obs, 'ps_true': ps_true,
    })

df = simulate_study(n=3000)
cov_cols = ['age','severity','bmi','smoke','female']
X = df[cov_cols].values
scaler = StandardScaler()
X_s = scaler.fit_transform(X)
T = df['treated'].values.astype(int)
Y = df['y_obs'].values
print(f"n={len(df)}, treated={T.sum()}, control={(~df.treated).sum()}")
print(f"True ATE = 3.000")

## 1. Horvitz-Thompson IPW Estimator

ATE_IPW = (1/n) Σ [ T_i·Y_i/e(X_i) − (1−T_i)·Y_i/(1−e(X_i)) ]

Equivalently, create Horvitz-Thompson weights: w_i = T_i/e(X_i) + (1−T_i)/(1−e(X_i)).

In [ ]:
def fit_propensity(X_s, T, method='logistic'):
    if method == 'logistic':
        m = LogisticRegression(max_iter=500, C=1.0, random_state=0)
    else:
        m = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=0)
    ps = cross_val_predict(m, X_s, T, cv=5, method='predict_proba')[:, 1]
    return np.clip(ps, 0.01, 0.99)   # clip to avoid extreme weights


def ipw_ate(Y, T, ps):
    """Horvitz-Thompson IPW estimator for ATE."""
    treated = T.astype(bool)
    mu1_ipw = np.mean(Y[treated]  / ps[treated])
    mu0_ipw = np.mean(Y[~treated] / (1 - ps[~treated]))
    return mu1_ipw - mu0_ipw


def sipw_ate(Y, T, ps):
    """Stabilized IPW: normalize weights within each arm."""
    treated = T.astype(bool)
    w1 = 1 / ps[treated]
    w0 = 1 / (1 - ps[~treated])
    mu1 = np.average(Y[treated],  weights=w1)
    mu0 = np.average(Y[~treated], weights=w0)
    return mu1 - mu0


ps_lr = fit_propensity(X_s, T, method='logistic')
ps_gb = fit_propensity(X_s, T, method='gb')

print(f"{'Estimator':<30} {'ATE':>8}")
print(f"{'True ATE':<30} {'3.000':>8}")
print(f"{'Naïve DiM':<30} {Y[T==1].mean()-Y[T==0].mean():>8.3f}")
print(f"{'IPW (logistic PS)':<30} {ipw_ate(Y,T,ps_lr):>8.3f}")
print(f"{'SIPW (logistic PS)':<30} {sipw_ate(Y,T,ps_lr):>8.3f}")
print(f"{'SIPW (GB PS)':<30} {sipw_ate(Y,T,ps_gb):>8.3f}")

## 2. Weight Diagnostics — Extreme Weights

Extreme weights (very small e(X) for treated, or very small 1−e(X) for control) indicate near-violations of overlap and inflate variance dramatically. Always inspect the weight distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, ps, label in zip(axes, [ps_lr, ps_gb], ['Logistic', 'Gradient Boosting']):
    w = np.where(T==1, 1/ps, 1/(1-ps))
    ax.hist(w, bins=60, color='steelblue', alpha=0.7)
    ax.axvline(np.percentile(w, 99), color='red', ls='--',
               label=f'99th pctile = {np.percentile(w,99):.1f}')
    ax.set_xlabel('IPW weight')
    ax.set_ylabel('Count')
    ax.set_title(f'Weight Distribution — {label} PS')
    ax.legend()
    pct_extreme = (w > 10).mean()
    ax.set_title(f'Weight Distribution — {label} PS\n'
                 f'{pct_extreme:.1%} of weights > 10')

plt.tight_layout()
plt.savefig('04_weights.png', dpi=110, bbox_inches='tight')
plt.show()

## 3. Outcome Regression

Direct adjustment: fit μ̂₁(X) = E[Y|T=1, X] and μ̂₀(X) = E[Y|T=0, X] separately, then compute ATE = (1/n) Σ [μ̂₁(X_i) − μ̂₀(X_i)] over the full sample. No weighting needed, but biased if outcome model is misspecified.

In [ ]:
def outcome_regression_ate(Y, T, X_s):
    """Fit separate outcome models per arm via cross-fitting."""
    treated = T.astype(bool)
    # Fit on treated
    m1 = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=0)
    m1.fit(X_s[treated], Y[treated])
    mu1_hat = m1.predict(X_s)

    # Fit on control
    m0 = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=0)
    m0.fit(X_s[~treated], Y[~treated])
    mu0_hat = m0.predict(X_s)

    ate = (mu1_hat - mu0_hat).mean()
    return ate, mu1_hat, mu0_hat


ate_or, mu1_hat, mu0_hat = outcome_regression_ate(Y, T, X_s)
print(f"Outcome regression ATE = {ate_or:.3f}")

## 4. Augmented IPW (AIPW) — Doubly Robust

AIPW = (1/n) Σ [ μ̂₁(X_i) − μ̂₀(X_i) + T_i(Y_i − μ̂₁(X_i))/ê(X_i) − (1−T_i)(Y_i − μ̂₀(X_i))/(1−ê(X_i)) ]

The augmentation terms are **influence function corrections**: they debias the outcome model estimate using the propensity score residuals. If the outcome model is perfect, the corrections vanish. If the PS model is perfect, the corrections alone give IPW. Either one correct → consistent estimate.

In [ ]:
def aipw_ate(Y, T, ps, mu1_hat, mu0_hat):
    """
    Augmented IPW (doubly robust) estimator for ATE.
    Unbiased if either PS model or outcome model is correctly specified.
    """
    T = T.astype(float)
    ps = np.clip(ps, 0.01, 0.99)
    term1 = mu1_hat - mu0_hat
    term2 = T * (Y - mu1_hat) / ps
    term3 = (1 - T) * (Y - mu0_hat) / (1 - ps)
    psi = term1 + term2 - term3
    return psi.mean(), psi.std() / np.sqrt(len(Y))


# AIPW with well-specified models
ate_aipw, se_aipw = aipw_ate(Y, T, ps_lr, mu1_hat, mu0_hat)
ci_lo, ci_hi = ate_aipw - 1.96*se_aipw, ate_aipw + 1.96*se_aipw
print(f"AIPW ATE = {ate_aipw:.3f} ± {se_aipw:.3f}  95% CI [{ci_lo:.3f}, {ci_hi:.3f}]")

In [ ]:
# Double robustness: what happens if one model is wrong?
def misspecified_ps(X_s, T):
    """Intentionally bad PS — intercept-only model."""
    return np.full(len(T), T.mean())

def misspecified_outcome(Y, T, X_s):
    """Intentionally bad outcome model — predict grand mean."""
    mu1 = np.full(len(Y), Y[T==1].mean())
    mu0 = np.full(len(Y), Y[T==0].mean())
    return mu1, mu0

ps_bad     = misspecified_ps(X_s, T)
mu1_bad, mu0_bad = misspecified_outcome(Y, T, X_s)

results = {
    'True ATE':                          3.000,
    'Naïve DiM':                         Y[T==1].mean()-Y[T==0].mean(),
    'IPW (good PS)':                     sipw_ate(Y, T, ps_lr),
    'OR (good outcome)':                 ate_or,
    'AIPW (both good)':                  aipw_ate(Y, T, ps_lr, mu1_hat, mu0_hat)[0],
    'AIPW (bad PS, good outcome)':       aipw_ate(Y, T, ps_bad, mu1_hat, mu0_hat)[0],
    'AIPW (good PS, bad outcome)':       aipw_ate(Y, T, ps_lr, mu1_bad, mu0_bad)[0],
    'AIPW (both bad)':                   aipw_ate(Y, T, ps_bad, mu1_bad, mu0_bad)[0],
}

print(f"{'Estimator':<40} {'ATE':>8}")
print("-" * 50)
for k, v in results.items():
    flag = ' ← BIASED' if abs(v - 3.0) > 0.3 else ''
    print(f"{k:<40} {v:>8.3f}{flag}")

In [ ]:
# Visualize doubly robust property
labels = list(results.keys())
vals   = list(results.values())
colors = ['#4CAF50' if abs(v-3) < 0.3 else '#F44336' for v in vals]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels, vals, color=colors, alpha=0.8, edgecolor='white')
ax.axvline(3.0, color='black', lw=2, label='True ATE = 3.0')
ax.axvspan(2.7, 3.3, alpha=0.1, color='green', label='±10% of truth')
ax.set_xlabel('ATE Estimate')
ax.set_title('Double Robustness: AIPW stays consistent if either model is correct')
ax.legend()
plt.tight_layout()
plt.savefig('04_double_robustness.png', dpi=110, bbox_inches='tight')
plt.show()